# General GRPO Training Pipeline

This notebook implements the general SFT -> GRPO training pipeline for the Data Pipeline Incident Response agent. It works with any Hugging Face causal LM.
Set the configuration variables in the next cell to get started.


In [ ]:
import json
import os
import sys
import textwrap
from typing import Any, Dict, List, Optional
import torch
import numpy as np
from datasets import Dataset
from trl import SFTTrainer, GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported

# Ensure repo root is on path if running locally
sys.path.insert(0, os.path.abspath('.'))
from src.environment import DataPipelineEnv
from src.models import PipelineAction, PipelineObservation

# --- Configuration ---
MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct'
SFT_EPOCHS = 3
GRPO_EPOCHS = 2
LORA_RANK = 16
KL_COEFF = 0.1
NUM_GENERATIONS = 4
SFT_TASKS = ['easy', 'medium']
GRPO_TASKS = ['hard', 'hard2']
OUTPUT_DIR = './checkpoints'
SKIP_SFT = False
PUSH_TO_HUB = None
LOAD_IN_4BIT = True

# Optional: Set your Hugging Face token here if not in environment
HF_TOKEN = os.getenv('HF_TOKEN', '')


In [ ]:
SYSTEM_PROMPT = textwrap.dedent('''
You are an expert data engineer diagnosing and fixing broken data pipelines.
You receive pipeline state and must choose ONE action per turn.

WORKFLOW:
1. read_data_sample on raw tables to see the data.
2. check_schema / compare_schema if schema drift is suspected.
3. Apply fix: add_data_filter or patch_transformation.
4. run_pipeline to verify.
5. alert_upstream_team if data is genuinely corrupted.

RULES:
- Respond with ONLY a JSON object. No markdown, no prose.
- Never apply a fix before reading the data.
- Never repeat the same action that didn't work.
- If a "unique" assertion fails, use dedup.
- After parse_currency, always chain coalesce.
''').strip()

GOLD_ACTIONS = {
    'easy': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_orders', 'n_rows': 20}},
        {'action_type': 'add_data_filter', 'params': {'step_id': 'transform_orders', 'filter_condition': 'user_id IS NOT NULL'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'medium': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_order_items', 'n_rows': 20}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_items', 'patch_type': 'dedup', 'column': 'order_item_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
    ],
    'hard': [
        {'action_type': 'read_data_sample', 'params': {'table': 'raw_ads_insights', 'n_rows': 20}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_ads_insights'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'parse_currency', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'coalesce', 'column': 'spend'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_insights', 'patch_type': 'parse_currency', 'column': 'impressions'}},
        {'action_type': 'add_data_filter', 'params': {'step_id': 'transform_insights', 'filter_condition': 'impressions IS NOT NULL'}},
        {'action_type': 'run_pipeline', 'params': {}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_conversions', 'patch_type': 'dedup', 'column': 'event_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
        {'action_type': 'compare_schema', 'params': {'table': 'raw_conversions'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_conversions', 'patch_type': 'strip_prefix', 'column': 'campaign_id'}},
        {'action_type': 'patch_transformation', 'params': {'step_id': 'transform_conversions', 'patch_type': 'cast_column', 'column': 'campaign_id'}},
        {'action_type': 'run_pipeline', 'params': {}},
        {'action_type': 'alert_upstream_team', 'params': {'team': 'meta_ads_api_team', 'issue_description': 'Graph API outage: N/A impressions'}},
    ],
}

def format_obs_for_training(obs: PipelineObservation, step: int) -> str:
    failed = '\n'.join(f'  [{r.assertion_id}] {r.assertion_type} on {r.table}({r.column or "N/A"}): {r.actual}' for r in obs.failed_assertions) or '  (none)'
    passed = ', '.join(r.assertion_id for r in obs.passed_assertions) or 'none'
    dag = '\n'.join(f'  {n.step_id}: {n.input_table} -> {n.output_table}' + (f' | filters: {n.applied_filters}' if n.applied_filters else '') + (f' | patches: {n.applied_patches}' if n.applied_patches else '') for n in obs.dag_structure)
    hist = '\n'.join(f'  {r.date}: {r.status} ({r.row_count} rows)' for r in obs.historical_runs)
    schema = ''
    if obs.current_schema: schema += '\nCURRENT SCHEMA: ' + json.dumps(obs.current_schema)
    if obs.schema_diff: schema += '\nSCHEMA DIFF: ' + json.dumps(obs.schema_diff)
    sample = ''
    if obs.data_sample: sample = '\nDATA SAMPLE: ' + json.dumps(obs.data_sample[:3], default=str)
    actions = '\n'.join(f'  {a}' for a in obs.actions_taken[-4:]) or '  (none)'
    return textwrap.dedent(f'''
    STEP {step}/{obs.max_steps} | TASK: {obs.task_id} ({obs.difficulty})
    DESCRIPTION: {obs.description}
    PIPELINE PASSED: {obs.pipeline_passed}
    LAST ACTION RESULT: {obs.last_action_result}
    DAG:\n{dag}
    FAILING:\n{failed}
    PASSING: {passed}
    HISTORY:\n{hist}
    RECENT ACTIONS:\n{actions}
    {sample}{schema}
    Respond with exactly ONE action JSON object.
    ''').strip()

def collect_gold_trajectories(task_ids: List[str], n_episodes: int = 10) -> List[tuple]:
    pairs = []
    for task_id in task_ids:
        gold = GOLD_ACTIONS.get(task_id, [])
        if not gold: continue
        for _ in range(n_episodes):
            env = DataPipelineEnv(task_id=task_id)
            obs = env.reset()
            for step_idx, action_dict in enumerate(gold, 1):
                obs_text = format_obs_for_training(obs, step_idx)
                action = PipelineAction(**action_dict)
                pairs.append((obs_text, json.dumps(action_dict)))
                result = env.step(action)
                obs = result.observation
                if obs.pipeline_passed: break
    return pairs


In [ ]:
def parse_action_from_text(text: str) -> Optional[PipelineAction]:
    import re
    text = re.sub(r'<think>[\s\S]*?</think>', '', text, flags=re.DOTALL).strip()
    if '```' in text:
        lines = text.split('\n')
        text = '\n'.join(l for l in lines if not l.strip().startswith('```')).strip()
    start = text.find('{')
    if start == -1: return None
    end = text.rfind('}') + 1
    if end <= start: return None
    try:
        data = json.loads(text[start:end])
        if 'action_type' in data: return PipelineAction(**data)
    except: pass
    return None

def pipeline_reward_fn(completions, prompts=None, **kwargs) -> list:
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0].get('content', '')
        action = parse_action_from_text(text)
        if action is None:
            rewards.append(-0.3)
            continue
        reward = 0.3
        try:
            env = DataPipelineEnv(task_id='hard')
            obs = env.reset()
            result = env.step(action)
            reward += result.reward or 0.0
            if action.action_type == 'compare_schema':
                if result.observation.schema_diff and len(result.observation.schema_diff) > 0:
                    reward += 0.3
        except:
            reward -= 0.2
        rewards.append(float(reward))
    return rewards


## Load Model

In [ ]:
print('\n[1/5] Loading model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token=HF_TOKEN,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)')


## Supervised Fine-Tuning (SFT)

In [ ]:
sft_dir = os.path.join(OUTPUT_DIR, 'sft')
if not SKIP_SFT:
    print(f'\n[2/5] Collecting SFT trajectories from {SFT_TASKS}...')
    gold_pairs = collect_gold_trajectories(SFT_TASKS, n_episodes=10)
    print(f'  Collected {len(gold_pairs)} (observation, action) pairs')

    sft_texts = [
        tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT},
             {'role': 'user', 'content': obs_text},
             {'role': 'assistant', 'content': action_json}],
            tokenize=False, add_generation_prompt=False,
        )
        for obs_text, action_json in gold_pairs
    ]

    sft_dataset = Dataset.from_dict({'text': sft_texts})
    print(f'  SFT dataset: {len(sft_dataset)} examples')

    sft_trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=sft_dataset,
        dataset_text_field='text',
        max_seq_length=4096,
        args=TrainingArguments(
            report_to="none",
            average_tokens_across_devices=False,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=SFT_EPOCHS,
            warmup_ratio=0.1,
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=1,
            optim='adamw_8bit',
            weight_decay=0.01,
            lr_scheduler_type='cosine',
            output_dir=sft_dir,
            save_steps=50,
            seed=42,
        ),
    )
    print('\n[3/5] Training SFT...')
    sft_stats = sft_trainer.train()
    print(f'  SFT complete. Loss: {sft_stats.training_loss:.4f}')
    model.save_pretrained(sft_dir)
    tokenizer.save_pretrained(sft_dir)
else:
    print('\n[2/5] Skipping SFT (--skip-sft)')
    print('[3/5] Skipping SFT training')


## GRPO Training

In [ ]:
print(f'\n[4/5] Building GRPO prompts from {GRPO_TASKS}...')
from src.tasks import TASKS as available_tasks
valid_grpo = [t for t in GRPO_TASKS if t in available_tasks]
if not valid_grpo:
    print(f'  WARNING: No valid GRPO tasks found. Available: {list(available_tasks.keys())}')
    valid_grpo = ['hard']

grpo_prompts = []
for task_id in valid_grpo:
    for _ in range(25):
        env = DataPipelineEnv(task_id=task_id)
        obs = env.reset()
        prompt_text = format_obs_for_training(obs, step=1)
        chat_prompt = tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM_PROMPT},
             {'role': 'user', 'content': prompt_text}],
            tokenize=False, add_generation_prompt=True,
        )
        grpo_prompts.append({'prompt': chat_prompt})

grpo_dataset = Dataset.from_list(grpo_prompts)
print(f'  GRPO dataset: {len(grpo_dataset)} prompts')

grpo_dir = os.path.join(OUTPUT_DIR, 'grpo')
grpo_config = GRPOConfig(
    report_to="none",
    average_tokens_across_devices=False,
    output_dir=grpo_dir,
    num_generations=NUM_GENERATIONS,
    max_completion_length=200,
    temperature=0.8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=GRPO_EPOCHS,
    learning_rate=5e-5,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    beta=KL_COEFF,
    loss_type='grpo',
    logging_steps=1,
    save_steps=50,
    seed=42,
    max_prompt_length=4096,
    max_grad_norm=1.0,
    warmup_steps=1,
)

grpo_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    reward_funcs=pipeline_reward_fn,
    args=grpo_config,
    train_dataset=grpo_dataset,
)

print(f'  KL coeff: {KL_COEFF}, G: {NUM_GENERATIONS}')
print('\n  Starting GRPO training...')
grpo_stats = grpo_trainer.train()
print(f'  GRPO complete. Loss: {grpo_stats.training_loss:.4f}')
model.save_pretrained(grpo_dir)
tokenizer.save_pretrained(grpo_dir)


## Evaluation

In [ ]:
print('\n[5/5] Evaluating trained model...')
FLM.for_inference(model)

def run_eval_episode(model, tokenizer, task_id: str, max_steps: int = 20) -> dict:
    env = DataPipelineEnv(task_id=task_id)
    obs = env.reset()
    rewards = []
    step = 0
    for step in range(1, max_steps + 1):
        if obs.pipeline_passed:
            break
        prompt_text = format_obs_for_training(obs, step)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt_text},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, return_tensors='pt', add_generation_prompt=True
        ).to(model.device)
        with torch.no_grad():
            out_ids = model.generate(
                inputs, max_new_tokens=200, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(out_ids[0][inputs.shape[1]:], skip_special_tokens=True).strip()
        action = parse_action_from_text(response)
        if action is None:
            action = PipelineAction(action_type='compare_schema', params={'table': 'raw_orders'})
        result = env.step(action)
        obs = result.observation
        rewards.append(result.reward or 0.0)
        if result.done:
            break
    n_total = len(obs.failed_assertions) + len(obs.passed_assertions)
    n_passed = len(obs.passed_assertions)
    score = min(max(n_passed / n_total if n_total > 0 else 0, 0.01), 0.99)
    return {
        'task_id': task_id,
        'score': round(score, 3),
        'pipeline_passed': obs.pipeline_passed,
        'total_reward': round(sum(rewards), 3),
        'steps': step,
    }

eval_tasks = [t for t in ['easy', 'medium', 'hard', 'hard2'] if t in available_tasks]
baseline = {'easy': 0.70, 'medium': 0.55, 'hard': 0.30, 'hard2': 0.30}
print(f"\n{'Task':<10} {'Baseline':<10} {'Trained':<10} {'Delta':<10} {'Pass?'}")
print('=' * 50)
for task_id in eval_tasks:
    r = run_eval_episode(model, tokenizer, task_id=task_id)
    base = baseline.get(task_id, 0.0)
    delta = r['score'] - base
    status = '[PASSED]' if r['pipeline_passed'] else '[FAILED]'
    sign = '+' if delta >= 0 else ''
    print(f"{task_id:<10} {base:<10.3f} {r['score']:<10.3f} {sign}{delta:<9.3f} {status}")


## Push to Hugging Face Hub

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    print(f'\nPushing to HF Hub: {PUSH_TO_HUB}')
    model.push_to_hub_merged(
        PUSH_TO_HUB, tokenizer,
        save_method='merged_16bit', token=HF_TOKEN,
    )
    print(f'Model pushed to https://huggingface.co/{PUSH_TO_HUB}')
else:
    print('Skipping push to Hub (PUSH_TO_HUB or HF_TOKEN not set).')
